In [15]:
import numpy as np
import torch as t

from model import Config
from pipeline import OptimizerSpec, Trainer
from plots import plot_curves, plot_final_grid

import os


t.manual_seed(0)
np.random.seed(0)


ce que dois encore faire : 

- les hook
- analysed de fourier sur les activation
- connection a wandb si on a envie..

In [16]:


#! JE RUN SUR METAL DONC SUPPORTE PAS FLOAT64, DU COUP J'AI MIS UN CAST EN FLOAT32 DANS LA CROSS ENTROPY POUR EVITER LES LOSS SPIKES ET GRADIENTS POURRIS. A REVERIFIER SI ON VEUT RUN SUR GPU CLASSIQUE GENRE CUDA ET VOIR CE QUE CA CHANGE

config = Config(
    p=113,
    d_model=128,
    d_mlp=512,          
    num_heads=4,
    n_ctx=3,
    act_type='ReLU', #! a ce niveau la j'ai pas mis de modularité
    frac_train=0.3,
    num_epochs=40_000,
    seed=0,
)

specs = [
    OptimizerSpec('adamw',
                  lr=1e-3,
                  weight_decay=1.0,
                  extra={'betas': (0.9, 0.98)}), #! repris meme quand dans la baseline de jean
]

print(f"Device: {config.device}")
print(f"Specs:  {specs[0].describe()}")

Device: mps
Specs:  adamw(ALL, lr=0.001, wd=1.0, betas=(0.9, 0.98))


In [10]:
trainer = Trainer(
    config,
    specs,
    seed=0,
    label='nanda_baseline',
    eval_every=50,
    fourier_every=None,    #! Fourier désactivé pour l'instant, pas encore fonctionnel !!
    warmup_steps=10,
    verbose_every=2000,
    verbose_build=True,
)

history = trainer.fit()

print()
print(f"final train_acc = {history['train_acc'][-1]:.3f}")
print(f"final test_acc  = {history['test_acc'][-1]:.3f}")


=== Trainer 'nanda_baseline' (seed=0) ===
  spec: adamw(ALL, lr=0.001, wd=1.0, betas=(0.9, 0.98))
Spec order after sorting:
  0 : adamw(ALL, lr=0.001, wd=1.0, betas=(0.9, 0.98))
  adamw(ALL, lr=0.001, wd=1.0, betas=(0.9, 0.98)): 11 tensors, 226,816 params
      |__ embed.W_E
      |__ pos_embed.W_pos
      |__ blocks.0.attn.W_K
      |__ blocks.0.attn.W_Q
      |__ blocks.0.attn.W_V
      |__ blocks.0.attn.W_O
      |__ blocks.0.mlp.W_in
      |__ blocks.0.mlp.b_in
      |__ blocks.0.mlp.W_out
      |__ blocks.0.mlp.b_out
      |__ unembed.W_U
  [nanda_baseline seed=0] epoch     0 | train acc 0.009 | test acc 0.009
  [nanda_baseline seed=0] epoch  2000 | train acc 1.000 | test acc 0.065
  [nanda_baseline seed=0] epoch  4000 | train acc 1.000 | test acc 0.080
  [nanda_baseline seed=0] epoch  6000 | train acc 1.000 | test acc 0.109
  [nanda_baseline seed=0] epoch  8000 | train acc 1.000 | test acc 0.194
  [nanda_baseline seed=0] epoch 10000 | train acc 1.000 | test acc 0.989
  [nanda_ba

In [11]:
plot_curves(history, title="Nanda baseline — AdamW (lr=1e-3, wd=1.0)")

In [12]:
plot_final_grid(trainer, title="Modular addition table (mod 113) learned by AdamW")

In [ ]:
# save
trainer.save_run('runs/nanda_baseline_seed0')


In [ ]:
# reload
# Reconstruit Config, OptimizerSpec, modèle, optimizers et history.
trainer_reloaded = Trainer.from_run('runs/nanda_baseline_seed0')

print(f"restored epoch : {trainer_reloaded.epoch}")
print(f"final test_acc : {trainer_reloaded.history['test_acc'][-1]:.3f}")
print(f"device         : {trainer_reloaded.config.device}")
print(f"specs          : {trainer_reloaded.specs[0].describe()}")